# Conditionally clean ancillae: reasoning, circuit reconstruction, and verification

**Question.** Can we replace the clean fifth output of Image #2 with the first wire of Images #1/#3, then use that wire to control another quantum operation?

**Original variant ($d=0,e=0$).** The proposed replacement fails: its output is $r=\overline a\oplus f$, rather than $f$. A correct four-wire computation can instead control the later operation directly from two intermediate wires.

**New variant ($d=1,e=0$), added in Section 9.** The reused wire is valid conditional workspace, but the later consumer must retain the flag: $d\,r_+=g=abc$. We recreate both updated compute circuits and verify the guarded compute–use–uncompute construction. The earlier sections retain the original variant for comparison. The general verification guide now appears in Section 10.

This is an executable research notebook. All quantum simulations are local and ideal; no IBM account or hardware is used. Saved outputs include the circuit drawings and checks. Run all cells in order to reproduce them. An output-free companion, `conditionally_clean_ancilla_analysis_source.ipynb`, is provided for editing.

The notebook has three levels of evidence:

1. A Boolean derivation that identifies the condition under which a borrowed wire is known.
2. Exhaustive, phase-sensitive statevector checks of all allowed basis inputs.
3. A comparison of the complete **compute–use–uncompute** maps, including restoration of workspace and behavior on superpositions.

The analytical statements hold exactly for the ideal gates. Numerical assertions use absolute tolerance $10^{-10}$ and zero relative tolerance; they are numerical verification, not a symbolic formal proof.

## 1. Setup, conventions, and sources

Select this repository's Python environment as the notebook kernel. The required packages are already included in its synthesis, visualization, and notebook extras. The optional Aer example uses the simulation extra. This notebook does not install or synchronize packages.

Human wire numbers run from **1 at the top**. Qiskit indices start at **0**. Throughout the mathematics, $|a,b,c,d\rangle$ lists wires **top to bottom**. Qiskit's printed bitstrings run in the reverse order: `q[n-1] ... q[0]`. We explicitly convert between these conventions instead of reading bitstrings by eye.

| Human wire | Qiskit index | Initial role |
|---|---|---|
| 1 | 0 | Input $a$ |
| 2 | 1 | Input $b$ |
| 3 | 2 | Input $c$ |
| 4 | 3 | Clean workspace, initially zero |
| 5 in Image #2 | 4 | Clean output, initially zero |

Our theoretical source is Section 3, Definition 3.1, and Figure 1 of [Khattar and Gidney, *Rise of conditionally clean ancillae for efficient quantum circuit constructions*](https://quantum-journal.org/papers/q-2025-05-21-1752/), especially pages 4–7. The [published version](https://quantum-journal.org/papers/q-2025-05-21-1752/) provides the same reference.

The green “On” boxes in the supplied images are **conditional probability displays**. They neither prepare qubits nor measure/postselect the physical state. We recreate the quantum gates and evaluate the displayed condition separately below.

In [ ]:
from itertools import product
from importlib.metadata import version
from collections.abc import Callable
import sys
from io import BytesIO

import numpy as np
import matplotlib

matplotlib.use("Agg")
import matplotlib.pyplot as plt
from IPython.display import Image, Markdown, display
from qiskit import QuantumCircuit, transpile
from qiskit.circuit import Gate
from qiskit.circuit.library import CCXGate, UGate, XGate
from qiskit.quantum_info import Operator, Statevector, partial_trace, state_fidelity

ATOL = 1e-10
np.set_printoptions(precision=6, suppress=True)
print("Python:", sys.version.split()[0])
for package in ("qiskit", "numpy", "matplotlib", "nbformat", "nbclient"):
    print(f"{package}: {version(package)}")


def table(headers: list[str], rows: list[list[object]]) -> None:
    """Display a small Markdown table without a dataframe dependency."""
    lines = ["| " + " | ".join(headers) + " |"]
    lines.append("| " + " | ".join("---" for _ in headers) + " |")
    lines.extend("| " + " | ".join(map(str, row)) + " |" for row in rows)
    display(Markdown("\n".join(lines)))


def basis_state(bits: tuple[int, ...]) -> Statevector:
    """Build a basis state from bits in top-to-bottom wire order."""
    index = sum(bit << wire for wire, bit in enumerate(bits))
    amplitudes = np.zeros(2 ** len(bits), dtype=complex)
    amplitudes[index] = 1
    return Statevector(amplitudes)


def assert_amplitudes(actual: Statevector, expected: Statevector) -> None:
    """Compare all complex amplitudes without dropping any phases."""
    np.testing.assert_allclose(actual.data, expected.data, atol=ATOL, rtol=0)


def show_figure(fig: plt.Figure) -> None:
    """Embed a PNG so saved outputs display without an interactive backend."""
    buffer = BytesIO()
    fig.savefig(buffer, format="png", dpi=150, bbox_inches="tight")
    display(Image(data=buffer.getvalue()))
    plt.close(fig)


def draw(circuit: QuantumCircuit, title: str, initial_state: bool = False) -> None:
    """Render a circuit using the images' one-based human wire numbers."""
    fig = circuit.draw(
        output="mpl",
        fold=-1,
        initial_state=initial_state,
        style={"name": "bw", "fontsize": 11, "subfontsize": 9},
        with_layout=False,
    )
    for text in fig.axes[0].texts:
        label = text.get_text()
        for i in range(circuit.num_qubits):
            label = label.replace(rf"${{q}}_{{{i}}}$", f"q{i + 1}")
        text.set_text(label)
    fig.suptitle(title, fontsize=13, y=1.02)
    show_figure(fig)


print("Bit order example: top-to-bottom 1000 has Qiskit dictionary key 0001.")
print(basis_state((1, 0, 0, 0)).to_dict())

## 2. Recreate the two circuits

Image #2 contains a Toffoli from wires 1 and 3 into wire 4, a CNOT from wire 2 into wire 3, and a Toffoli with **two open controls** on wires 3 and 4 into wire 5.

Images #1 and #3 are the same proposed four-wire circuit: the last target moves to wire 1, with an $X$ on that wire immediately beforehand. `CCXGate(ctrl_state="00")` expresses both open controls directly. The probability-display column is omitted because it is not a physical gate.

We separate input preparation from computation. The drawings include the three Hadamards. Basis-state checks below omit them to test the computation on all possible inputs.

In [ ]:
def reference_compute() -> QuantumCircuit:
    """Recreate Image #2 after input preparation."""
    qc = QuantumCircuit(5, name="reference")
    qc.ccx(0, 2, 3)
    qc.cx(1, 2)
    qc.append(CCXGate(ctrl_state="00"), [2, 3, 4])
    return qc


def proposed_compute() -> QuantumCircuit:
    """Recreate Images #1/#3 after input preparation."""
    qc = QuantumCircuit(4, name="proposed")
    qc.ccx(0, 2, 3)
    qc.cx(1, 2)
    qc.x(0)
    qc.append(CCXGate(ctrl_state="00"), [2, 3, 0])
    return qc


def with_hadamards(compute: QuantumCircuit) -> QuantumCircuit:
    """Prepare the three-input uniform superposition shown in the images."""
    qc = QuantumCircuit(compute.num_qubits)
    qc.h([0, 1, 2])
    qc.compose(compute, inplace=True)
    return qc


reference = reference_compute()
proposed = proposed_compute()
draw(with_hadamards(reference), "Original reference: e=0, d=0 (Image #2)", initial_state=True)
draw(
    with_hadamards(proposed),
    "Original reuse: e=0, d=0 (Images #1/#3)",
    initial_state=True,
)

## 3. What “conditionally clean” means here

Immediately after the first Toffoli,

$$
|a,b,c,0\rangle\longmapsto |a,b,c,d\rangle,\qquad d=ac.
$$

The condition $d=1$ has a unique pair of original controls: $a=c=1$. Thus both controls are known to be $|1\rangle$ on that branch. On $d=0$, the pairs $(a,c)$ can be $00,01,10$; neither is fixed.

A projector makes the meaning precise. With $P=|1\rangle\langle1|$ on wire 4, the projected state has wires 1 and 3 fixed to $|1\rangle$. This is a statement about a subspace, not an instruction to measure the flag. The simulation below normalizes projected branches only to inspect this invariant; the circuit itself remains coherent.

A borrowed wire need not be known outside the condition. If it is used as temporary clean workspace, the result must be consumed with the appropriate condition, and the temporary operations must be reversed before restoring the original computation. The predicate that certifies cleanliness must remain valid during that use.

In [ ]:
def conditional_branch(
    state: Statevector, wire: int, value: int
) -> tuple[float, Statevector]:
    """Inspect a projected branch without adding a measurement to the circuit."""
    indices = np.arange(len(state.data))
    mask = ((indices >> wire) & 1) == value
    projected = np.where(mask, state.data, 0)
    probability = float(np.vdot(projected, projected).real)
    if probability < ATOL:
        raise ValueError("The requested branch has zero probability.")
    return probability, Statevector(projected / np.sqrt(probability))


first = QuantumCircuit(4)
first.h([0, 1, 2])
first.ccx(0, 2, 3)
after_first = Statevector.from_instruction(first)
rows = []
for d in (0, 1):
    weight, branch = conditional_branch(after_first, 3, d)
    p_a = branch.probabilities(qargs=[0])[1]
    p_c = branch.probabilities(qargs=[2])[1]
    rows.append([d, f"{weight:.6f}", f"{p_a:.6f}", f"{p_c:.6f}"])
    expected = 1.0 if d else 1 / 3
    np.testing.assert_allclose([p_a, p_c], [expected, expected], atol=ATOL, rtol=0)
table(["d", "Branch probability", "P(q1=1 given d)", "P(q3=1 given d)"], rows)

The two “On” displays correspond to the $d=1$ row. They do not say that these wires are globally initialized to 1.

Next, the CNOT changes wire 3 to $e=b\oplus c$. On $d=1$, wire 1 is still 1, but wire 3 is now $1\oplus b$. An invariant belongs to a specific point in the circuit; it must be updated after gates change the relevant wires.

After the $X$ in the proposed circuit, wire 1 is zero on $d=1$. Unfortunately the final Toffoli fires on $d=0$ and $e=0$. It uses the branch where this zero-initialization guarantee is unavailable.

## 4. Derive both outputs before simulating

Use multiplication for Boolean AND, $\oplus$ for XOR, and an overline for NOT. Define

$$
d=ac,\qquad e=b\oplus c,\qquad f=\overline d\,\overline e.
$$

The five-wire reference produces

$$
|a,b,c,0,0\rangle\longmapsto |a,b,e,d,f\rangle.
$$

A Toffoli XORs its control predicate into its target; it does not overwrite the target. In the proposed circuit the target starts at $a\oplus1$, so

$$
r=a\oplus1\oplus f=\overline a\oplus f,
\qquad
|a,b,c,0\rangle\longmapsto |r,b,e,d\rangle.
$$

Therefore $r=f$ when the original $a=1$, and $r=\overline f$ when $a=0$. In particular, original input $000$ gives $f=1$ and $r=0$.

These expressions are derived from the gate semantics independently of Qiskit. They are the reference against which the simulator will be checked.

In [ ]:
def boolean_values(a: int, b: int, c: int) -> tuple[int, int, int, int]:
    """Return d, e, f, r from the independently derived Boolean expressions."""
    d = a & c
    e = b ^ c
    f = (1 - d) & (1 - e)
    r = a ^ 1 ^ f
    return d, e, f, r


def predicate(a: int, b: int, c: int) -> int:
    """Return the desired condition for the later operation."""
    return (1 - (a & c)) & (1 - (b ^ c))


rows = []
for a, b, c in product((0, 1), repeat=3):
    d, e, f, r = boolean_values(a, b, c)
    rows.append([f"{a}{b}{c}", d, e, f, r, "yes" if f == r else "NO"])
table(["abc", "d", "e", "f: reference", "r: proposed", "Match?"], rows)

## 5. Exactly how Qiskit double-checks the derivation

For each of the eight values of $abc$, we prepare a basis vector with all workspace wires at zero. `Statevector.evolve(circuit)` computes its output. We build an independent expected vector from the expressions above, then compare **every complex amplitude** with `numpy.testing.assert_allclose`.

This checks the complete output, not just the intended result wire. It checks phase as well as bit values. We deliberately do **not** use a separate global-phase-insensitive comparison for each basis state: input-dependent phases matter on superpositions.

The two output spaces have different dimensions, so directly comparing `Operator(reference)` with `Operator(proposed)` is not meaningful. First we verify each circuit against its own full output map, then compare the promised result values. Later we compare their complete logical use under an explicit workspace contract.

In [ ]:
mismatches = []
reference_columns = []
proposed_columns = []
for a, b, c in product((0, 1), repeat=3):
    d, e, f, r = boolean_values(a, b, c)
    actual_reference = basis_state((a, b, c, 0, 0)).evolve(reference)
    actual_proposed = basis_state((a, b, c, 0)).evolve(proposed)
    assert_amplitudes(actual_reference, basis_state((a, b, e, d, f)))
    assert_amplitudes(actual_proposed, basis_state((r, b, e, d)))
    reference_columns.append(actual_reference.data)
    proposed_columns.append(actual_proposed.data)
    if f != r:
        mismatches.append(f"{a}{b}{c}")

assert mismatches == ["000", "001", "010", "011"]
print("PASS: all 8 complete reference outputs match the derived map.")
print("PASS: all 8 complete proposed outputs match the derived map.")
print("The proposed replacement FAILS on these inputs:", mismatches)
print("Assertions confirm the counterexample; the failure is the circuit's claim.")

Why does this establish behavior on arbitrary superpositions? If a linear map agrees on every allowed basis vector **with the same phase convention**, it agrees on their linear combinations:

$$
\sum_{abc}\alpha_{abc}|a,b,c,0,0\rangle
\longmapsto
\sum_{abc}\alpha_{abc}|a,b,e,d,f\rangle.
$$

The analogous statement for the proposal replaces the ket on the right by $|r,b,e,d\rangle$. The source Hadamards select the special case $\alpha_{abc}=1/\sqrt8$; they do not restrict what must be checked for reuse inside a larger algorithm.

As an additional illustration, the next cell prepares a reproducible complex superposition and checks these maps directly. This randomized example is a smoke test; the exhaustive basis-amplitude comparison is the stronger evidence.

In [ ]:
rng = np.random.default_rng(20260917)
alpha = rng.normal(size=8) + 1j * rng.normal(size=8)
alpha /= np.linalg.norm(alpha)
inputs = list(product((0, 1), repeat=3))
for circuit, columns in ((reference, reference_columns), (proposed, proposed_columns)):
    initial = np.zeros(2**circuit.num_qubits, dtype=complex)
    for coefficient, (a, b, c) in zip(alpha, inputs):
        initial[a + 2 * b + 4 * c] = coefficient
    actual = Statevector(initial).evolve(circuit)
    expected = Statevector(np.column_stack(columns) @ alpha)
    assert_amplitudes(actual, expected)
    print(
        circuit.name,
        "complex-superposition fidelity:",
        state_fidelity(actual, expected),
    )

## 6. Why matching probability displays is not enough

For the shown Hadamard inputs, three of eight assignments have $f=1$, and three have $r=1$:

$$
\Pr(f=1)=\Pr(r=1)=\frac38.
$$

These are different sets of assignments. Equal output frequencies do not show that the output is correlated with the right input branches.

We also inspect the one-wire density matrices. `partial_trace(state, qargs)` discards the wires listed in `qargs`; therefore we list **all wires except the output wire**. This directly reproduces the density-matrix check used in the earlier explanation.

In [ ]:
state_reference = Statevector.from_instruction(with_hadamards(reference))
state_proposed = Statevector.from_instruction(with_hadamards(proposed))
rho_f = partial_trace(state_reference, [0, 1, 2, 3]).data
rho_r = partial_trace(state_proposed, [1, 2, 3]).data
np.testing.assert_allclose(rho_f, [[5 / 8, 0], [0, 3 / 8]], atol=ATOL, rtol=0)
np.testing.assert_allclose(rho_r, [[5 / 8, 1 / 4], [1 / 4, 3 / 8]], atol=ATOL, rtol=0)
print("Reference output density matrix:\n", rho_f)
print("Proposed output density matrix:\n", rho_r)

fig, axes = plt.subplots(1, 3, figsize=(11, 3.2), layout="constrained")
x = np.arange(2)
axes[0].bar(x - 0.17, rho_f.diagonal().real, 0.34, label="Reference f")
axes[0].bar(x + 0.17, rho_r.diagonal().real, 0.34, label="Proposed r")
axes[0].set(
    xticks=x,
    xticklabels=["0", "1"],
    ylim=(0, 1),
    title="Equal Z-basis probabilities",
    ylabel="Probability",
)
axes[0].legend(fontsize=9)
for ax, rho, title in zip(
    axes[1:], [rho_f, rho_r], ["Reference: Re(rho_f)", "Proposed: Re(rho_r)"]
):
    ax.imshow(rho.real, cmap="Blues", vmin=0, vmax=0.7)
    ax.set(xticks=[0, 1], yticks=[0, 1], title=title)
    for i, j in product(range(2), repeat=2):
        ax.text(
            j,
            i,
            f"{rho[i, j].real:.3f}",
            ha="center",
            va="center",
            color="white" if rho[i, j].real > 0.4 else "black",
        )
show_figure(fig)
X = np.array([[0, 1], [1, 0]])
print("X expectation, reference:", np.trace(rho_f @ X).real)
print("X expectation, proposed: ", np.trace(rho_r @ X).real)

The off-diagonal entries differ. Measuring in the X basis would distinguish the reduced outputs: the reference has $\langle X\rangle=0$, while the proposal has $\langle X\rangle=1/2$. Even matching reduced density matrices would not by itself prove equality of correlations with other wires; here, however, the mismatch is already visible locally.

## 7. Use the result to control a later operation

Let $U$ act on a **separate target register**. The desired logical action preserves the input and applies $U$ precisely when $f=1$:

$$
|a,b,c\rangle|\psi\rangle\longmapsto
|a,b,c\rangle U^{f(a,b,c)}|\psi\rangle.
$$

In the reference construction we compute $f$ in wire 5, use it to control $U$, and reverse the computation. In the corrected construction we compute $d$ and $e$, apply $U$ with open controls on both, then reverse the CNOT and first Toffoli. This replaces a stored output with direct controls:

$$
\overline d\,\overline e=f.
$$

The four-wire figure counts the three inputs and one workspace wire. The target of $U$ is additional in **both** constructions. For a one-qubit $U$, the complete circuits below therefore use six and five total wires respectively.

We choose a gate with mixing and complex phases, $U=\operatorname{U}(0.7,0.2,-0.4)$, so the check tests more than a classical target flip. You can replace `u_gate` with another one-qubit unitary gate and rerun the checks. The reasoning extends to a disjoint multi-qubit target register, but the helper code below explicitly assumes one target qubit.

In [ ]:
u_gate = UGate(0.7, 0.2, -0.4)


def use_stored_result(
    compute: QuantumCircuit, output_wire: int, gate: Gate
) -> QuantumCircuit:
    """Compute a result, use it to control a one-qubit gate, and uncompute."""
    n = compute.num_qubits
    qc = QuantumCircuit(n + 1)
    qc.compose(compute, qubits=list(range(n)), inplace=True)
    qc.barrier()
    qc.append(gate.control(1), [output_wire, n])
    qc.barrier()
    qc.compose(compute.inverse(), qubits=list(range(n)), inplace=True)
    return qc


def direct_control(gate: Gate) -> QuantumCircuit:
    """Use e=0 and d=0 directly, then restore the four computation wires."""
    qc = QuantumCircuit(5)
    qc.ccx(0, 2, 3)
    qc.cx(1, 2)
    qc.barrier()
    qc.append(gate.control(2, ctrl_state="00"), [2, 3, 4])
    qc.barrier()
    qc.cx(1, 2)
    qc.ccx(0, 2, 3)
    return qc


reference_use = use_stored_result(reference, 4, u_gate)
corrected_use = direct_control(u_gate)
wrong_use = use_stored_result(proposed, 0, u_gate)
draw(reference_use, "Reference: compute f, control U, uncompute (target on q6)")
draw(corrected_use, "Correct: control U directly with e=0 and d=0 (target on q5)")
draw(wrong_use, "Incorrect predicate: use r as the control, then uncompute")

### Compare logical maps and check workspace restoration

A six-qubit unitary and a five-qubit unitary cannot be compared directly. Their **logical maps** can: both have inputs $a,b,c$ and a target bit $t$, with all auxiliary wires initialized to zero.

For each of these 16 logical basis inputs, we simulate the output and collect two parts:

- The amplitudes with workspace still at zero, ordered by the four logical wires $(a,b,c,t)$.
- The probability of any workspace bit being nonzero (“leakage” from the promised output subspace).

This produces a $16\times16$ logical matrix. We do not silently discard dirty workspace: a nonzero leakage is a verification failure. The independent expected matrix is built from the Boolean predicate and the $2\times2$ matrix of $U$, without synthesizing another copy of the same circuit.

Equality of these matrices in a fixed phase convention, together with zero leakage, checks all superpositions and inputs entangled with an external reference by linearity. It is more demanding than a few sampled inputs.

In [ ]:
def logical_matrix(circuit: QuantumCircuit) -> tuple[np.ndarray, float]:
    """Extract the clean-workspace map on a,b,c and the last physical wire."""
    n = circuit.num_qubits
    target = n - 1
    workspace = list(range(3, target))
    logical_to_physical = [(x & 7) | (((x >> 3) & 1) << target) for x in range(16)]
    indices = np.arange(2**n)
    dirty = np.zeros(2**n, dtype=bool)
    for wire in workspace:
        dirty |= ((indices >> wire) & 1).astype(bool)
    matrix = np.zeros((16, 16), dtype=complex)
    max_leakage = 0.0
    for column, physical_index in enumerate(logical_to_physical):
        state = Statevector.from_int(physical_index, 2**n).evolve(circuit)
        matrix[:, column] = state.data[logical_to_physical]
        max_leakage = max(max_leakage, float(np.sum(np.abs(state.data[dirty]) ** 2)))
    return matrix, max_leakage


def expected_logical_matrix(
    gate: Gate, condition: Callable[[int, int, int], int]
) -> np.ndarray:
    """Build the desired logical action directly from its specification."""
    result = np.zeros((16, 16), dtype=complex)
    unitary = Operator(gate).data
    for column in range(16):
        a, b, c, t = ((column >> i) & 1 for i in range(4))
        action = unitary if condition(a, b, c) else np.eye(2)
        for out_t in (0, 1):
            row = a + 2 * b + 4 * c + 8 * out_t
            result[row, column] = action[out_t, t]
    return result


expected = expected_logical_matrix(u_gate, predicate)
logical_results = {}
rows = []
for name, circuit in [
    ("Reference", reference_use),
    ("Corrected direct controls", corrected_use),
    ("Proposed r control", wrong_use),
]:
    matrix, leakage = logical_matrix(circuit)
    logical_results[name] = matrix
    error = float(np.max(np.abs(matrix - expected)))
    rows.append([name, circuit.num_qubits, f"{leakage:.2e}", f"{error:.2e}"])
    assert leakage < ATOL
    if name != "Proposed r control":
        np.testing.assert_allclose(matrix, expected, atol=ATOL, rtol=0)
    else:
        assert not np.allclose(matrix, expected, atol=ATOL, rtol=0)
        wrong_predicate = lambda a, b, c: a ^ 1 ^ predicate(a, b, c)
        np.testing.assert_allclose(
            matrix, expected_logical_matrix(u_gate, wrong_predicate), atol=ATOL, rtol=0
        )

table(
    [
        "Construction",
        "Total wires",
        "Max workspace leakage",
        "Max amplitude error vs f-controlled U",
    ],
    rows,
)
print("PASS: reference and corrected construction implement the desired logical map.")
print("The proposed r control restores workspace but implements the WRONG predicate.")

Workspace restoration alone does not establish correctness: the incorrect construction also reverses its computation, but it has already applied $U$ on the wrong branches.

A particularly clear counterexample sets $abc=000$, target $t=0$, and $U=X$. The desired operation flips the target to 1. The proposed $r$-controlled operation leaves it at 0. The next cell checks that explicitly, then tests the corrected complex-valued $U$ on an arbitrary coherent logical input.

In [ ]:
wrong_x, leakage = logical_matrix(use_stored_result(proposed, 0, XGate()))
correct_x, _ = logical_matrix(direct_control(XGate()))
initial_0000 = basis_state((0, 0, 0, 0)).data
assert np.argmax(np.abs(correct_x @ initial_0000)) == 8
assert np.argmax(np.abs(wrong_x @ initial_0000)) == 0
print("abc=000, target=0: correct X target = 1; proposed r-controlled X target = 0.")

logical_input = rng.normal(size=16) + 1j * rng.normal(size=16)
logical_input /= np.linalg.norm(logical_input)
actual = Statevector(logical_results["Corrected direct controls"] @ logical_input)
ideal = Statevector(expected @ logical_input)
assert_amplitudes(actual, ideal)
print("Corrected circuit coherent-input fidelity:", state_fidelity(actual, ideal))

## 8. A valid example of reusing a former Toffoli control

The gate arrangement itself is not prohibited. Let $d=ac$ as before. Apply $X$ on wire 1, then a Toffoli controlled by $d$ and $b$ into wire 1. Its temporary value is

$$
s=\overline a\oplus db.
$$

If $d=1$, then $a=1$ and $s=b$. For all inputs,

$$
ds=db=abc.
$$

Therefore using **both $d$ and $s$** to control $U$ implements the predicate $abc$. Reverse the workspace Toffoli and $X$, then uncompute $d$. The complete circuit is identity on the target outside the active predicate, and all borrowed input wires are restored. Although the temporary $X$ acts outside $d=1$ too, the later use is guarded and the temporary changes are undone.

This is a valid conditional-workspace example, but it computes a **different predicate** from the one in your images. It explains why a positive control on the flag can participate safely, while the proposed negative-control use does not acquire the needed initialization guarantee.

In [ ]:
def valid_borrowing(gate: Gate) -> QuantumCircuit:
    """Use q1 as conditional workspace, consuming s only together with d."""
    qc = QuantumCircuit(5)
    qc.ccx(0, 2, 3)
    qc.x(0)
    qc.ccx(3, 1, 0)
    qc.barrier()
    qc.append(gate.control(2), [3, 0, 4])
    qc.barrier()
    qc.ccx(3, 1, 0)
    qc.x(0)
    qc.ccx(0, 2, 3)
    return qc


borrowed_use = valid_borrowing(u_gate)
draw(borrowed_use, "Valid borrowing: consume s together with d, then restore")
borrowed_matrix, leakage = logical_matrix(borrowed_use)
expected_abc = expected_logical_matrix(u_gate, lambda a, b, c: a & b & c)
np.testing.assert_allclose(borrowed_matrix, expected_abc, atol=ATOL, rtol=0)
assert leakage < ATOL
print(
    "PASS: all 16 logical basis columns implement abc-controlled U; workspace restored."
)

## 9. New variant: the final Toffoli fires on d = 1 and e = 0

We now change the final Toffoli in **both** original circuits: wire 3, carrying $e$, remains an **open control**, while wire 4, carrying $d$, becomes a **closed control**. The $X$ before the final Toffoli in the four-wire circuit remains in place. The earlier sections and diagrams retain the original $d=0,e=0$ variant for comparison.

This change makes the temporary use of wire 1 valid **on the $d=1$ branch**. It does not make that wire a globally clean output. To use its result to control a later operation, retain a closed control on $d$ as well.

Write the new reference predicate as $g$ to distinguish it from the original $f$:

$$
g=d\,\overline e=ac\,\overline{b\oplus c}=abc.
$$

The last equality holds because $d=1$ forces $a=c=1$, and then $e=b\oplus1=\overline b$. Thus the new reference output is 1 only on original input $abc=111$.

The new reused result is

$$
r_+=\overline a\oplus g.
$$

On $d=1$, $a=1$, so $r_+=g=\overline e$. On $d=0$, $g=0$ and $r_+=\overline a$, which may be 1. Consequently,

$$
\boxed{d\,r_+=g,\qquad r_+\ne g\text{ in general}.}
$$

This identity is the correct contract for consuming the borrowed result. The target of the first Toffoli may control the second Toffoli here; the positive flag selects the branch where the reused target was zeroed by $X$.

### 9.1 Updated compute circuits and mixed-control bit order

For the control list `[2, 3]`, Qiskit treats index 2 (wire 3, $e$) as the low bit of `ctrl_state`, and index 3 (wire 4, $d$) as its high bit. Therefore `ctrl_state="10"` means **$e=0,d=1$**, not the reverse. We check the control gate independently on every value of $e,d$ and its target before using it in the reconstructed circuits.

In [ ]:
# Verify the mixed-control encoding independently of the surrounding circuit.
mixed_control = QuantumCircuit(3)
mixed_control.append(CCXGate(ctrl_state="10"), [0, 1, 2])
for e, d, target in product((0, 1), repeat=3):
    actual = basis_state((e, d, target)).evolve(mixed_control)
    expected_target = target ^ (d & (1 - e))
    assert_amplitudes(actual, basis_state((e, d, expected_target)))
print('PASS: ctrl_state="10" on [e, d] fires exactly when e=0 and d=1.')


def positive_flag_reference() -> QuantumCircuit:
    """Compute g = d AND NOT e in a separate clean fifth wire."""
    qc = QuantumCircuit(5, name="positive_flag_reference")
    qc.ccx(0, 2, 3)
    qc.cx(1, 2)
    qc.append(CCXGate(ctrl_state="10"), [2, 3, 4])
    return qc


def positive_flag_reused() -> QuantumCircuit:
    """Compute the conditionally valid r_plus on the former first control."""
    qc = QuantumCircuit(4, name="positive_flag_reused")
    qc.ccx(0, 2, 3)
    qc.cx(1, 2)
    qc.x(0)
    qc.append(CCXGate(ctrl_state="10"), [2, 3, 0])
    return qc


positive_reference = positive_flag_reference()
positive_reused = positive_flag_reused()
draw(
    with_hadamards(positive_reference),
    "New reference: open e control, closed d control; output g",
    initial_state=True,
)
draw(
    with_hadamards(positive_reused),
    "New reuse: open e control, closed d control; output r+",
    initial_state=True,
)

### 9.2 Compare every basis input, including the condition used to consume it

The two complete compute maps are

$$
|a,b,c,0,0\rangle\longmapsto |a,b,e,d,g\rangle,
\qquad
|a,b,c,0\rangle\longmapsto |r_+,b,e,d\rangle.
$$

The following table compares the clean output $g$, the unguarded borrowed output $r_+$, and the guarded predicate $d\,r_+$. The assertions compare each complete statevector against its derived map, not merely these output bits.

In [ ]:
def positive_flag_predicate(a: int, b: int, c: int) -> int:
    """Specify the new predicate directly from the original input bits."""
    return a & b & c


positive_rows = []
positive_mismatches = []
for a, b, c in product((0, 1), repeat=3):
    d = a & c
    e = b ^ c
    g = d & (1 - e)
    r_plus = (1 - a) ^ g
    guarded = d & r_plus
    assert g == positive_flag_predicate(a, b, c)
    assert guarded == g
    assert_amplitudes(
        basis_state((a, b, c, 0, 0)).evolve(positive_reference),
        basis_state((a, b, e, d, g)),
    )
    assert_amplitudes(
        basis_state((a, b, c, 0)).evolve(positive_reused),
        basis_state((r_plus, b, e, d)),
    )
    positive_rows.append(
        [
            f"{a}{b}{c}",
            d,
            e,
            g,
            r_plus,
            guarded,
            "yes" if r_plus == g else "NO",
        ]
    )
    if r_plus != g:
        positive_mismatches.append(f"{a}{b}{c}")
assert positive_mismatches == ["000", "001", "010", "011"]
table(
    [
        "abc",
        "d",
        "e",
        "g: clean output",
        "r+: borrowed",
        "d AND r+",
        "r+ alone matches?",
    ],
    positive_rows,
)
print("PASS: both full compute maps match on all 8 basis inputs, including phases.")
print("PASS: d AND r+ equals g on every input.")
print("r+ alone still fails on:", positive_mismatches)

positive_reference_state = Statevector.from_instruction(
    with_hadamards(positive_reference)
)
positive_reused_state = Statevector.from_instruction(with_hadamards(positive_reused))
p_g = positive_reference_state.probabilities(qargs=[4])[1]
p_r_plus = positive_reused_state.probabilities(qargs=[0])[1]
# The last index is 11 for qargs=[0, 3]: both r+ and d equal 1.
p_guarded = positive_reused_state.probabilities(qargs=[0, 3])[3]
np.testing.assert_allclose(
    [p_g, p_r_plus, p_guarded], [1 / 8, 5 / 8, 1 / 8], atol=ATOL, rtol=0
)
table(
    ["Uniform-input event", "Probability"],
    [
        ["g=1", f"{p_g:.3f}"],
        ["r+=1", f"{p_r_plus:.3f}"],
        ["d=1 AND r+=1", f"{p_guarded:.3f}"],
    ],
)

Unlike the original negative-flag example, the uniform-input output probabilities now differ visibly: the clean output is 1 with probability $1/8$, while the borrowed wire is 1 with probability $5/8$. Guarding the borrowed result by $d=1$ selects exactly the correct branches and has probability $1/8$.

For original input $000$, the new reference has $g=0$, while the reused wire has $r_+=1$. A later operation controlled only by that reused wire would incorrectly fire. Adding the closed control on $d$ suppresses this error because $d=0$ on that input.

### 9.3 Updated compute–use–uncompute circuits

There are two correct ways to save the fifth computation wire for a later operation $U$ on a separate target:

1. **Consume the borrowed result with its flag.** Compute $r_+$ with the updated four-wire circuit, apply $U$ with closed controls on **both wire 1 ($r_+$) and wire 4 ($d$)**, then reverse the entire computation.
2. **Control directly on the intermediate values.** Compute $d$ and $e$, apply $U$ with an open control on wire 3 ($e$) and a closed control on wire 4 ($d$), then undo the CNOT and first Toffoli. This avoids computing $r_+$ at all.

The inverse must restore the borrowed wire before uncomputing the first Toffoli; otherwise that first gate would read a modified original control. We use the full circuit inverse in the borrowed-result construction so the order is explicit and correct.

Both methods implement

$$
|a,b,c,0\rangle|\psi\rangle
\longmapsto
|a,b,c,0\rangle U^{abc}|\psi\rangle.
$$

The original clean-output reference uses one more workspace qubit and restores it as well. The diagrams below omit input Hadamards so the logical operation can be tested on arbitrary inputs. As before, the target of $U$ is additional to the four or five computation wires.

In [ ]:
def positive_flag_guarded_use(gate: Gate) -> QuantumCircuit:
    """Consume r_plus only together with d, then restore every borrowed wire."""
    compute = positive_flag_reused()
    qc = QuantumCircuit(5)
    qc.compose(compute, qubits=[0, 1, 2, 3], inplace=True)
    qc.barrier()
    qc.append(gate.control(2), [0, 3, 4])
    qc.barrier()
    qc.compose(compute.inverse(), qubits=[0, 1, 2, 3], inplace=True)
    return qc


def positive_flag_direct_use(gate: Gate) -> QuantumCircuit:
    """Control directly on e=0 and d=1, then restore the computation wires."""
    qc = QuantumCircuit(5)
    qc.ccx(0, 2, 3)
    qc.cx(1, 2)
    qc.barrier()
    qc.append(gate.control(2, ctrl_state="10"), [2, 3, 4])
    qc.barrier()
    qc.cx(1, 2)
    qc.ccx(0, 2, 3)
    return qc


positive_reference_use = use_stored_result(positive_reference, 4, u_gate)
positive_guarded_use = positive_flag_guarded_use(u_gate)
positive_direct_use = positive_flag_direct_use(u_gate)
positive_unguarded_use = use_stored_result(positive_reused, 0, u_gate)
draw(positive_reference_use, "New reference: compute g, control U, uncompute")
draw(positive_guarded_use, "Valid reuse: control U with BOTH r+ and d, then uncompute")
draw(positive_direct_use, "Direct use: control U with e=0 and d=1, then uncompute")

### 9.4 Verify the complete controlled operations

We reuse the phase-sensitive logical-matrix check from Section 7. For every logical basis input $(a,b,c,t)$ we compare the complete output against the independent specification $g=abc$, and check that workspace returns to zero.

For contrast, we also test the incorrect circuit that controls $U$ only with $r_+$. It restores workspace but applies the wrong operation on the four inputs with $a=0$.

In [ ]:
positive_expected = expected_logical_matrix(u_gate, positive_flag_predicate)
positive_use_rows = []
positive_use_matrices = {}
for name, circuit, should_match in [
    ("New clean-output reference", positive_reference_use, True),
    ("Borrowed result guarded by d", positive_guarded_use, True),
    ("Direct e=0, d=1 controls", positive_direct_use, True),
    ("Borrowed result WITHOUT d", positive_unguarded_use, False),
]:
    matrix, leakage = logical_matrix(circuit)
    positive_use_matrices[name] = matrix
    error = float(np.max(np.abs(matrix - positive_expected)))
    assert leakage < ATOL
    if should_match:
        np.testing.assert_allclose(matrix, positive_expected, atol=ATOL, rtol=0)
    else:
        assert not np.allclose(matrix, positive_expected, atol=ATOL, rtol=0)
        wrong_condition = lambda a, b, c: (1 - a) ^ (a & b & c)
        np.testing.assert_allclose(
            matrix, expected_logical_matrix(u_gate, wrong_condition), atol=ATOL, rtol=0
        )
    positive_use_rows.append(
        [
            name,
            circuit.num_qubits,
            f"{leakage:.2e}",
            f"{error:.2e}",
            "PASS" if should_match else "REJECT",
        ]
    )
table(
    [
        "Construction",
        "Total wires",
        "Max workspace leakage",
        "Max amplitude error",
        "g-controlled U?",
    ],
    positive_use_rows,
)

positive_rng = np.random.default_rng(20260918)
positive_input = positive_rng.normal(size=16) + 1j * positive_rng.normal(size=16)
positive_input /= np.linalg.norm(positive_input)
positive_ideal_state = Statevector(positive_expected @ positive_input)
positive_actual_state = Statevector(
    positive_use_matrices["Borrowed result guarded by d"] @ positive_input
)
assert_amplitudes(positive_actual_state, positive_ideal_state)
print("PASS: guarded borrowing also agrees on a seeded complex superposition.")

positive_bad_x, _ = logical_matrix(use_stored_result(positive_reused, 0, XGate()))
positive_good_x, _ = logical_matrix(positive_flag_guarded_use(XGate()))
zero_logical_input = basis_state((0, 0, 0, 0)).data
assert np.argmax(np.abs(positive_bad_x @ zero_logical_input)) == 8
assert np.argmax(np.abs(positive_good_x @ zero_logical_input)) == 0
print(
    "Counterexample 000, target=0: unguarded r+ flips X incorrectly; d-guarded r+ does not."
)

The new predicate also simplifies to $abc$, so a three-controlled $U$ on the original input wires is another logical description. It needs no explicit intermediate $d$ or $e$ wire at that level; the cost and ancillary requirements after decomposition depend on $U$ and the available gate set. This observation is separate from whether the borrowed-output construction is correct.

**Conclusion for the changed Toffoli:** the reused wire correctly stores the needed intermediate result **conditional on $d=1$**. It can replace the separate output in the complete operation when the consumer uses **both $d$ and $r_+$**, followed by uncomputation. Merely changing the final Toffoli's control polarity does not make an unguarded use of $r_+$ correct.

## 10. Ways to verify future circuits

Start by stating the contract: allowed inputs, required initial workspace, logical wire order, desired action, and which wires must be restored or may be discarded. Choose a method that actually checks that contract.

| Goal | Method | What it establishes and what it misses |
|---|---|---|
| Reversible Boolean logic | Exhaustive truth table plus phase check | Complete for small allowed input sets. An independent Boolean specification prevents checking a circuit against a copy of itself. |
| One particular quantum input | `Statevector.evolve`, amplitudes, or fidelity | Checks that input only; random states are useful smoke tests, not a universal proof. |
| Small equal-width unitaries | `Operator` matrices or `Operator.equiv` | Tests the whole unitary, after aligning wire order. `equiv` permits one global phase. |
| Known-zero workspace | Compare restricted input columns / logical isometries and check leakage | Tests the relevant subspace without incorrectly demanding equality for dirty workspace inputs. This is the method in Section 7. |
| Output subsystem or discarded garbage | `partial_trace`; for a universal claim compare induced channels | A reduced state check covers only the tested input. Include correlations with a reference when specifying channel equivalence. |
| Noisy circuits or reset | `SuperOp`, `Choi`, process fidelity; Aer density-matrix simulation | Compare channels, not unitary matrices. Specify whether measurement outcomes are retained, averaged, or classically fed forward. |
| Larger structured circuits | Decision diagrams or ZX-based equivalence tools | Can avoid dense matrices on favorable instances. Respect supported gates, layout, ancillary inputs, output/garbage annotations, and inconclusive results. |

The next cells give executable examples for phase checks, operator comparison after transpilation, and optional Aer statevector simulation. All dense methods scale exponentially: a statevector has $2^n$ entries and a unitary matrix has $4^n$. The small examples here are deliberately inexpensive.

### 10.1 Do not lose relative phase when checking basis inputs

`Statevector.equiv` accepts equality up to global phase. It is appropriate for comparing one physical pure state. Applying it separately to each basis input can incorrectly accept a gate with input-dependent phases.

The identity and $Z$ act identically on computational-basis probabilities, and their outputs are individually equivalent up to global phase on both basis inputs. They are different operators: on $|+\rangle$, $Z$ produces $|-\rangle$.

Reference: [Qiskit Statevector documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Statevector).

In [ ]:
identity = QuantumCircuit(1)
phase_error = QuantumCircuit(1)
phase_error.z(0)
for bit in (0, 1):
    state = basis_state((bit,))
    assert state.evolve(identity).equiv(state.evolve(phase_error))
print("Basis-by-basis equiv checks misleadingly accept I versus Z.")
plus = Statevector.from_label("+")
assert not plus.evolve(identity).equiv(plus.evolve(phase_error))
assert not Operator(identity).equiv(Operator(phase_error))
print("A superposition or whole-operator comparison correctly rejects I versus Z.")

### 10.2 Check a transpiled circuit against the original

When circuit width and wire order agree, compare their complete operators. `Operator.equiv` tolerates a **single** overall global phase. If a circuit will later itself be placed under coherent control, track that phase too: a formerly global phase becomes relative between the active and inactive control branches.

For an ancilla-assisted implementation, use the restricted-subspace comparison from Section 7 instead of expecting equality on all possible workspace states. For device transpilation, account for initial/final layout permutations before comparing matrices. The example below uses no coupling map and preserves the wire order.

Reference: [Qiskit Operator documentation](https://quantum.cloud.ibm.com/docs/en/api/qiskit/qiskit.quantum_info.Operator).

In [ ]:
compiled = transpile(
    corrected_use, basis_gates=["u", "cx"], optimization_level=2, seed_transpiler=7
)
original_operator = Operator(corrected_use)
compiled_operator = Operator(compiled)
assert original_operator.equiv(compiled_operator, atol=ATOL, rtol=0)
print("PASS: compiled circuit is equivalent up to a single global phase.")
print("Original operations:", dict(corrected_use.count_ops()))
print("Compiled operations:", dict(compiled.count_ops()))
print(
    "Fixed-phase maximum matrix difference:",
    np.max(np.abs(original_operator.data - compiled_operator.data)),
)

### 10.3 Use Aer for larger simulation workloads

Aer supplies statevector, density-matrix, and matrix-product-state methods, among others. A density-matrix method can represent mixed states and noise; matrix-product-state performance depends on entanglement and truncation settings. A Clifford stabilizer simulator is not a general exact simulator for Toffoli circuits. Check the supported method and gate set for your problem.

The following optional cell uses `save_statevector()` and verifies that Aer agrees with the earlier Qiskit statevector calculation. It does not infer correctness from sampled counts. Finite-shot measurement statistics are useful experimental evidence but cannot generally establish coherent circuit equivalence.

Reference: [Aer simulator documentation](https://qiskit.github.io/qiskit-aer/stubs/qiskit_aer.AerSimulator.html).

In [ ]:
try:
    from qiskit_aer import AerSimulator
except ImportError:
    print("Optional Aer check skipped: qiskit-aer is not installed.")
else:
    print("qiskit-aer:", version("qiskit-aer"))
    simulator = AerSimulator(method="statevector")
    aer_circuit = with_hadamards(reference)
    aer_circuit.save_statevector()
    aer_compiled = transpile(aer_circuit, simulator, seed_transpiler=7)
    result = simulator.run(aer_compiled, shots=1, seed_simulator=7).result()
    aer_state = Statevector(result.get_statevector())
    assert_amplitudes(aer_state, state_reference)
    print("PASS: Aer and Statevector agree on every complex amplitude.")

### 10.4 Tools beyond dense Qiskit simulation

**MQT QCEC** is a dedicated equivalence checker with decision-diagram and other checking strategies. It is a useful next step for comparing original and optimized circuits when constructing dense unitaries becomes expensive. Inspect the returned equivalence classification, not merely whether the program completed. Ancillary inputs, garbage outputs, and wire permutations must describe the intended contract. See the [official QCEC documentation](https://mqt.readthedocs.io/projects/qcec/en/stable/).

**PyZX** uses ZX-calculus circuit transformations and offers circuit equality verification. It is useful for proving equality after gate-level rewrites. An unsuccessful simplification-based check need not establish inequivalence; consult the method's return semantics. Dense tensor comparison, when used as a fallback, retains exponential cost. See [PyZX's documentation](https://pyzx.readthedocs.io/en/latest/) and [API reference](https://pyzx.readthedocs.io/en/latest/api.html).

**Boolean SAT/SMT or BDD methods** can check the reversible classical portion by encoding a “miter”: assert that at least one specified output differs between the candidate and the reference, under the required workspace assumptions. A satisfying assignment is a counterexample; an unsatisfiable result proves agreement of the encoded Boolean functions. This does not check arbitrary quantum phases. For phase-bearing or relative-phase Toffoli implementations, separately verify the phase function or use quantum equivalence methods.

These additional tools are suggestions, not installed or executed dependencies of this notebook. Use a separate compatible environment if their requirements differ from this repository's pinned Python/Qiskit stack.

### 10.5 A practical verification workflow

1. **Write the specification independently.** State the predicate or operator and the allowed workspace inputs.
2. **Normalize conventions.** Record endianness, wire permutations, open-control polarity, and whether preparation gates belong to the circuit being compared.
3. **Find small counterexamples first.** Test zero input, all-one input, branch boundaries, and all basis inputs when feasible.
4. **Check complex amplitudes.** Do not rely on populations, truth tables with unchecked phases, or per-input global-phase comparisons.
5. **Check the whole use.** Include the controlled consumer, inverse computation, and restoration of every borrowed wire. A test that only computes then uncomputes is insufficient.
6. **Choose the right equivalence notion.** Whole unitaries, clean-input isometries, and channels after discarding wires are different contracts.
7. **Scale deliberately.** Try structured simulation or an equivalence checker when dense methods become costly; record tolerances, seeds, gate conventions, and dependency versions.

For this example, the exhaustive amplitude checks establish the two derived compute maps numerically. The 16-column logical checks establish that the reference and corrected direct-control implementations agree with the desired $f$-controlled operation and restore workspace. The proposed reuse instead implements $r$-controlled behavior, even though its single-wire Z-basis probabilities happen to match on the uniform input.